# Forklift Semantic Segmentation Fine-tuning

`segmentation mask 1.1` のアノテーション zip を展開して、`labelmap.txt` に定義されたクラスで SegFormer-B0 を fine-tune します。学習後は `movie_preprocess_frames` から画像を選び、推論結果を `data/segmentation/predict/<ANNOTATION_DIR名>.zip` に CVAT import 可能な構造で保存します。

## 0. 依存パッケージ

In [ ]:
# 初回のみ必要に応じて実行してください。
%pip install torch torchvision transformers pillow tqdm ipywidgets scikit-learn


## 1. 設定

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import json
import math
import random
import shutil
import zipfile

import cv2
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
    from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "PyTorch / Transformers が見つかりません。先頭の依存パッケージセルを実行してください。"
    ) from error


DATA_DIR = Path("../data")
# None の場合は train ディレクトリ内の zip を辞書順で最初に選択します。
ANNOTATION_DIRNANE = None
TRAIN_ANNOTATION_BASE_DIR = DATA_DIR / "segmentation" / "train"


def annotation_zip_path_for_source(annotation_source_path: Path) -> Path:
    if annotation_source_path.suffix == ".zip":
        return annotation_source_path
    return annotation_source_path.parent / f"{annotation_source_path.name}.zip"


def validate_zip_members(zip_path: Path, extract_dir: Path) -> None:
    extract_root = extract_dir.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = (extract_root / member.filename).resolve()
            if not member_path.is_relative_to(extract_root):
                raise ValueError(f"Unsafe zip member path in {zip_path}: {member.filename}")


def extract_annotation_zip(annotation_zip_path: Path, extract_dir: Path) -> Path:
    validate_zip_members(annotation_zip_path, extract_dir)
    source_mtime = str(annotation_zip_path.stat().st_mtime_ns)
    marker_path = extract_dir / ".source_zip_mtime"
    needs_extract = not extract_dir.exists() or not marker_path.exists() or marker_path.read_text(encoding="utf-8") != source_mtime
    if needs_extract:
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(annotation_zip_path) as archive:
            archive.extractall(extract_dir)
        marker_path.write_text(source_mtime, encoding="utf-8")
    return find_annotation_root(extract_dir)


def find_annotation_root(extract_dir: Path) -> Path:
    if (extract_dir / "labelmap.txt").exists() and (extract_dir / "ImageSets" / "Segmentation" / "default.txt").exists():
        return extract_dir
    candidate_roots = [
        path.parent
        for path in extract_dir.rglob("labelmap.txt")
        if (path.parent / "ImageSets" / "Segmentation" / "default.txt").exists()
    ]
    if len(candidate_roots) == 1:
        return candidate_roots[0]
    if not candidate_roots:
        raise FileNotFoundError(f"Could not find CVAT annotation files under: {extract_dir}")
    raise ValueError(f"Multiple annotation roots found under {extract_dir}: {candidate_roots}")


def resolve_annotation_source_path(annotation_dirname: str | Path | None, train_annotation_base_dir: Path) -> Path:
    if annotation_dirname is None:
        zip_candidates = sorted(train_annotation_base_dir.glob("*.zip"))
        if not zip_candidates:
            raise FileNotFoundError(f"No annotation zip files found in: {train_annotation_base_dir}")
        return zip_candidates[0]
    annotation_source_path = Path(annotation_dirname)
    if not annotation_source_path.is_absolute():
        annotation_source_path = train_annotation_base_dir / annotation_source_path
    return annotation_source_path


def resolve_annotation_dir(annotation_source_path: Path) -> tuple[Path, Path | None]:
    source_candidates = [annotation_source_path]
    if annotation_source_path.suffix != ".zip":
        source_candidates.append(annotation_zip_path_for_source(annotation_source_path))
    source_path = next((path for path in source_candidates if path.exists()), None)
    if source_path is None:
        zip_candidates = sorted(annotation_source_path.parent.glob("*.zip"))
        if len(zip_candidates) == 1:
            source_path = zip_candidates[0]
        else:
            raise FileNotFoundError(f"Annotation directory or zip does not exist: {annotation_source_path}")
    if source_path.suffix == ".zip":
        annotation_dir = extract_annotation_zip(source_path, source_path.with_suffix(""))
        return annotation_dir, source_path
    return source_path, annotation_zip_path_for_source(source_path) if annotation_zip_path_for_source(source_path).exists() else None


ANNOTATION_SOURCE_PATH = resolve_annotation_source_path(ANNOTATION_DIRNANE, TRAIN_ANNOTATION_BASE_DIR)
ANNOTATION_DIR, ANNOTATION_ZIP_PATH = resolve_annotation_dir(ANNOTATION_SOURCE_PATH)
IMAGE_DIR = ANNOTATION_DIR / "JPEGImages"
IMAGE_SET_PATH = ANNOTATION_DIR / "ImageSets" / "Segmentation" / "default.txt"
SEGMENTATION_CLASS_DIR = ANNOTATION_DIR / "SegmentationClass"
LABELMAP_PATH = ANNOTATION_DIR / "labelmap.txt"

PREDICT_BASE_DIR = DATA_DIR / "segmentation" / "predict"
PREDICT_DIR = PREDICT_BASE_DIR / ANNOTATION_DIR.name
PREDICT_ZIP_PATH = PREDICT_BASE_DIR / f"{PREDICT_DIR.name}.zip"
PREDICT_SEGMENTATION_CLASS_DIR = PREDICT_DIR / "SegmentationClass"
PREDICT_SEGMENTATION_OBJECT_DIR = PREDICT_DIR / "SegmentationObject"
PREDICT_IMAGE_SET_PATH = PREDICT_DIR / "ImageSets" / "Segmentation" / "default.txt"

MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"
MODEL_OUTPUT_DIR = Path("../outputs/segformer_forklift_4class")

OTHER_CLASS_NAME = "other"


def load_labelmap_classes(labelmap_path: Path = LABELMAP_PATH) -> tuple[list[str], dict[str, tuple[int, int, int]]]:
    if not labelmap_path.exists():
        raise FileNotFoundError(f"Labelmap file does not exist: {labelmap_path}")

    class_names: list[str] = []
    class_colors: dict[str, tuple[int, int, int]] = {}
    for line_number, raw_line in enumerate(labelmap_path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        fields = line.split(":")
        if len(fields) < 2:
            raise ValueError(f"Invalid labelmap line {line_number}: {raw_line}")
        class_name = fields[0].strip()
        color_values = tuple(int(value.strip()) for value in fields[1].split(","))
        if not class_name:
            raise ValueError(f"Missing class name in labelmap line {line_number}: {raw_line}")
        if len(color_values) != 3 or any(value < 0 or value > 255 for value in color_values):
            raise ValueError(f"Invalid RGB color in labelmap line {line_number}: {raw_line}")
        if class_name in class_colors:
            raise ValueError(f"Duplicate class name in labelmap: {class_name}")
        class_names.append(class_name)
        class_colors[class_name] = color_values

    if not class_names:
        raise ValueError(f"No classes found in labelmap: {labelmap_path}")
    if OTHER_CLASS_NAME not in class_colors:
        raise ValueError(f"Labelmap must contain '{OTHER_CLASS_NAME}' class: {labelmap_path}")
    return class_names, class_colors


CLASS_NAMES, CLASS_COLORS = load_labelmap_classes()
CLASS_TO_ID = {class_name: class_id for class_id, class_name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {class_id: class_name for class_name, class_id in CLASS_TO_ID.items()}
COLOR_TO_CLASS = {color: class_name for class_name, color in CLASS_COLORS.items()}
LEGACY_OTHER_COLORS = list(dict.fromkeys([(0, 0, 0), (81, 185, 10), CLASS_COLORS[OTHER_CLASS_NAME]]))

RANDOM_SEED = 42
INPUT_SIZE = 512
BATCH_SIZE = 2
NUM_WORKERS = 0
VALIDATION_RATIO = 0.2
BLACK_COMPONENT_OTHER_PIXEL_THRESHOLD = 1000

HEAD_ONLY_EPOCHS = 5
FINETUNE_EPOCHS = 10
HEAD_LR = 1e-3
BACKBONE_LR = 1e-4
WEIGHT_DECAY = 0.01
CLASS_WEIGHTS = {class_name: 1.0 for class_name in CLASS_NAMES}
# if "object" in CLASS_WEIGHTS:
#     CLASS_WEIGHTS["object"] = 1.5

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


## 2. アノテーション確認とマスク正規化

In [ ]:
def read_image_ids_from_path(image_set_path: Path) -> list[str]:
    if not image_set_path.exists():
        return []
    return [line.strip() for line in image_set_path.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_image_ids_to_path(image_set_path: Path, image_ids: list[str]) -> None:
    image_set_path.parent.mkdir(parents=True, exist_ok=True)
    image_set_path.write_text("\n".join(image_ids) + "\n", encoding="utf-8")


def read_default_image_ids(image_set_path: Path = IMAGE_SET_PATH) -> list[str]:
    if not image_set_path.exists():
        raise FileNotFoundError(f"Image set file does not exist: {image_set_path}")
    return read_image_ids_from_path(image_set_path)


def extract_predict_zip_if_needed() -> None:
    if PREDICT_DIR.exists() or not PREDICT_ZIP_PATH.exists():
        return
    validate_zip_members(PREDICT_ZIP_PATH, PREDICT_DIR)
    PREDICT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PREDICT_ZIP_PATH) as archive:
        archive.extractall(PREDICT_DIR)
    if PREDICT_SEGMENTATION_OBJECT_DIR.exists():
        shutil.rmtree(PREDICT_SEGMENTATION_OBJECT_DIR)


def write_predict_annotation_zip() -> Path:
    if not PREDICT_DIR.exists():
        raise FileNotFoundError(f"Predict annotation directory does not exist: {PREDICT_DIR}")
    PREDICT_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_zip_path = PREDICT_ZIP_PATH.with_name(f"{PREDICT_ZIP_PATH.name}.tmp")
    if temporary_zip_path.exists():
        temporary_zip_path.unlink()
    with zipfile.ZipFile(temporary_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for source_path in sorted(PREDICT_DIR.rglob("*")):
            if source_path.is_dir():
                continue
            relative_path = source_path.relative_to(PREDICT_DIR)
            if relative_path.name in {".source_zip_mtime", "prediction_manifest.csv"}:
                continue
            if relative_path.parts and relative_path.parts[0] == "SegmentationObject":
                continue
            archive.write(source_path, relative_path.as_posix())
    temporary_zip_path.replace(PREDICT_ZIP_PATH)
    return PREDICT_ZIP_PATH


def cleanup_annotation_work_dirs() -> list[Path]:
    removed_paths: list[Path] = []
    if PREDICT_ZIP_PATH.exists() and PREDICT_DIR.exists():
        shutil.rmtree(PREDICT_DIR)
        removed_paths.append(PREDICT_DIR)
    if ANNOTATION_ZIP_PATH is not None:
        annotation_work_dir = ANNOTATION_ZIP_PATH.with_suffix("")
        if annotation_work_dir.exists() and (annotation_work_dir / ".source_zip_mtime").exists():
            shutil.rmtree(annotation_work_dir)
            removed_paths.append(annotation_work_dir)
    return removed_paths


def copy_annotation_dir_to_predict(preserve_existing_segmentation_masks: bool = True) -> None:
    if not ANNOTATION_DIR.exists():
        raise FileNotFoundError(f"Annotation directory does not exist: {ANNOTATION_DIR}")
    extract_predict_zip_if_needed()
    for source_path in sorted(ANNOTATION_DIR.rglob("*")):
        relative_path = source_path.relative_to(ANNOTATION_DIR)
        if relative_path.name == ".source_zip_mtime":
            continue
        if relative_path.parts and relative_path.parts[0] == "SegmentationObject":
            continue
        target_path = PREDICT_DIR / relative_path
        if source_path.is_dir():
            target_path.mkdir(parents=True, exist_ok=True)
            continue
        if (
            preserve_existing_segmentation_masks
            and target_path.exists()
            and relative_path.parts
            and relative_path.parts[0] in {"SegmentationClass", "SegmentationObject"}
        ):
            continue
        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, target_path)
    if PREDICT_SEGMENTATION_OBJECT_DIR.exists():
        shutil.rmtree(PREDICT_SEGMENTATION_OBJECT_DIR)


def prepare_predict_annotation_dir(preserve_existing_segmentation_masks: bool = True) -> None:
    extract_predict_zip_if_needed()
    existing_predict_ids = read_image_ids_from_path(PREDICT_IMAGE_SET_PATH)
    copy_annotation_dir_to_predict(preserve_existing_segmentation_masks=preserve_existing_segmentation_masks)
    merged_image_ids = read_image_ids_from_path(PREDICT_IMAGE_SET_PATH)
    for image_id in existing_predict_ids:
        if image_id not in merged_image_ids:
            merged_image_ids.append(image_id)
    if merged_image_ids:
        write_image_ids_to_path(PREDICT_IMAGE_SET_PATH, merged_image_ids)
    write_predict_annotation_zip()


def image_path_for_id(image_id: str) -> Path:
    return IMAGE_DIR / f"{image_id}.png"


def source_class_mask_path_for_id(image_id: str) -> Path:
    return SEGMENTATION_CLASS_DIR / f"{image_id}.png"


def class_mask_path_for_id(image_id: str) -> Path:
    return PREDICT_SEGMENTATION_CLASS_DIR / f"{image_id}.png"


def validate_labelmap_classes() -> None:
    missing_classes = [class_name for class_name in CLASS_NAMES if class_name not in CLASS_COLORS]
    if missing_classes:
        raise ValueError(f"Missing class colors: {missing_classes}")
    if OTHER_CLASS_NAME not in CLASS_TO_ID:
        raise ValueError(f"CLASS_NAMES must contain '{OTHER_CLASS_NAME}'")


def fill_small_black_components_with_neighbor_class(
    normalized_rgb: np.ndarray,
    black_mask: np.ndarray,
) -> tuple[np.ndarray, dict[str, int]]:
    stats = {
        "black_component_count": 0,
        "black_to_adjacent_pixel_count": 0,
        "black_to_other_pixel_count": 0,
    }
    if not black_mask.any():
        return normalized_rgb, stats

    component_count, component_labels = cv2.connectedComponents(black_mask.astype(np.uint8), connectivity=8)
    kernel = np.ones((3, 3), dtype=np.uint8)
    for component_label in range(1, component_count):
        component_mask = component_labels == component_label
        component_pixel_count = int(component_mask.sum())
        stats["black_component_count"] += 1

        fill_class_name = OTHER_CLASS_NAME
        if component_pixel_count < BLACK_COMPONENT_OTHER_PIXEL_THRESHOLD:
            neighbor_mask = cv2.dilate(component_mask.astype(np.uint8), kernel, iterations=1).astype(bool) & ~component_mask
            neighbor_counts: dict[str, int] = {}
            for class_name in CLASS_NAMES:
                if class_name == OTHER_CLASS_NAME:
                    continue
                color = np.array(CLASS_COLORS[class_name], dtype=np.uint8)
                neighbor_counts[class_name] = int((neighbor_mask & np.all(normalized_rgb == color, axis=2)).sum())
            adjacent_class_name, adjacent_count = max(neighbor_counts.items(), key=lambda item: item[1], default=(OTHER_CLASS_NAME, 0))
            if adjacent_count > 0:
                fill_class_name = adjacent_class_name

        normalized_rgb[component_mask] = np.array(CLASS_COLORS[fill_class_name], dtype=np.uint8)
        if fill_class_name == OTHER_CLASS_NAME:
            stats["black_to_other_pixel_count"] += component_pixel_count
        else:
            stats["black_to_adjacent_pixel_count"] += component_pixel_count
    return normalized_rgb, stats


def normalize_segmentation_class_mask(mask_rgb: np.ndarray) -> tuple[np.ndarray, dict[str, int]]:
    other_color = np.array(CLASS_COLORS[OTHER_CLASS_NAME], dtype=np.uint8)
    normalized_rgb = np.zeros_like(mask_rgb, dtype=np.uint8)
    normalized_rgb[:, :] = other_color
    stats: dict[str, int] = {"other_pixel_count": int(mask_rgb.shape[0] * mask_rgb.shape[1])}

    known_non_other_mask = np.zeros(mask_rgb.shape[:2], dtype=bool)
    for class_name in CLASS_NAMES:
        if class_name == OTHER_CLASS_NAME:
            continue
        color = np.array(CLASS_COLORS[class_name], dtype=np.uint8)
        class_mask = np.all(mask_rgb == color, axis=2)
        normalized_rgb[class_mask] = color
        known_non_other_mask |= class_mask
        stats[f"{class_name}_pixel_count"] = int(class_mask.sum())

    black_mask = np.all(mask_rgb == np.array((0, 0, 0), dtype=np.uint8), axis=2)
    normalized_rgb, black_stats = fill_small_black_components_with_neighbor_class(normalized_rgb, black_mask)

    legacy_other_mask = np.zeros(mask_rgb.shape[:2], dtype=bool)
    for legacy_color in LEGACY_OTHER_COLORS:
        if tuple(legacy_color) == (0, 0, 0):
            continue
        legacy_other_mask |= np.all(mask_rgb == np.array(legacy_color, dtype=np.uint8), axis=2)
    unexpected_mask = ~(known_non_other_mask | black_mask | legacy_other_mask)

    stats["black_pixel_count"] = int(black_mask.sum())
    stats.update(black_stats)
    stats["legacy_other_pixel_count"] = int(legacy_other_mask.sum())
    stats["unexpected_to_other_pixel_count"] = int(unexpected_mask.sum())
    stats["other_pixel_count"] = int(np.all(normalized_rgb == other_color, axis=2).sum())
    stats["changed_pixel_count"] = int(np.any(mask_rgb != normalized_rgb, axis=2).sum())
    return normalized_rgb, stats


def rgb_mask_to_class_ids(mask_rgb: np.ndarray) -> np.ndarray:
    normalized_rgb, _ = normalize_segmentation_class_mask(mask_rgb)
    class_id_mask = np.full(normalized_rgb.shape[:2], CLASS_TO_ID[OTHER_CLASS_NAME], dtype=np.int64)
    for class_name, class_id in CLASS_TO_ID.items():
        color = np.array(CLASS_COLORS[class_name], dtype=np.uint8)
        class_id_mask[np.all(normalized_rgb == color, axis=2)] = class_id
    return class_id_mask


def class_ids_to_rgb_mask(class_id_mask: np.ndarray) -> np.ndarray:
    rgb_mask = np.zeros((*class_id_mask.shape, 3), dtype=np.uint8)
    for class_id, class_name in ID_TO_CLASS.items():
        rgb_mask[class_id_mask == class_id] = np.array(CLASS_COLORS[class_name], dtype=np.uint8)
    return rgb_mask


def normalize_train_segmentation_class_masks_to_predict() -> pd.DataFrame:
    validate_labelmap_classes()
    prepare_predict_annotation_dir(preserve_existing_segmentation_masks=True)
    PREDICT_SEGMENTATION_CLASS_DIR.mkdir(parents=True, exist_ok=True)
    rows: list[dict[str, Any]] = []
    for mask_path in sorted(SEGMENTATION_CLASS_DIR.glob("*.png")):
        mask_rgb = np.array(Image.open(mask_path).convert("RGB"), dtype=np.uint8)
        normalized_rgb, stats = normalize_segmentation_class_mask(mask_rgb)
        normalized_mask_path = class_mask_path_for_id(mask_path.stem)
        Image.fromarray(normalized_rgb).save(normalized_mask_path)
        rows.append({
            "image_id": mask_path.stem,
            "source_mask_path": str(mask_path),
            "normalized_mask_path": str(normalized_mask_path),
            **stats,
        })
    write_predict_annotation_zip()
    return pd.DataFrame(rows)


default_image_ids = read_default_image_ids()
mask_normalization_df = normalize_train_segmentation_class_masks_to_predict()
annotated_image_ids = sorted(mask_normalization_df["image_id"].tolist()) if not mask_normalization_df.empty else []
missing_annotation_ids = sorted(set(default_image_ids) - set(annotated_image_ids))

print("default image count:", len(default_image_ids))
print("annotated SegmentationClass count:", len(annotated_image_ids))
print("missing SegmentationClass count:", len(missing_annotation_ids))
display(mask_normalization_df)


## 3. 学習データセット

In [ ]:
def build_annotated_records() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for image_id in annotated_image_ids:
        image_path = image_path_for_id(image_id)
        mask_path = class_mask_path_for_id(image_id)
        if not image_path.exists():
            print("Skip annotated mask because source image is missing:", image_id)
            continue
        if not mask_path.exists():
            print("Skip annotated mask because normalized mask is missing:", image_id)
            continue
        rows.append({
            "image_id": image_id,
            "image_path": str(image_path),
            "mask_path": str(mask_path),
        })
    return pd.DataFrame(rows)


def split_train_validation(records_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if len(records_df) <= 1:
        return records_df.copy(), records_df.copy()
    rng = np.random.default_rng(RANDOM_SEED)
    indices = np.arange(len(records_df))
    rng.shuffle(indices)
    validation_count = max(1, int(round(len(records_df) * VALIDATION_RATIO)))
    validation_indices = set(indices[:validation_count].tolist())
    train_rows = records_df.iloc[[idx for idx in range(len(records_df)) if idx not in validation_indices]].copy()
    validation_rows = records_df.iloc[[idx for idx in range(len(records_df)) if idx in validation_indices]].copy()
    return train_rows.reset_index(drop=True), validation_rows.reset_index(drop=True)


def apply_training_augmentations(image: Image.Image) -> Image.Image:
    if random.random() < 0.8:
        image = ImageEnhance.Brightness(image).enhance(random.uniform(0.75, 1.25))
    if random.random() < 0.8:
        image = ImageEnhance.Contrast(image).enhance(random.uniform(0.75, 1.35))
    if random.random() < 0.8:
        image = ImageEnhance.Color(image).enhance(random.uniform(0.75, 1.35))
    if random.random() < 0.4:
        image_hsv = np.array(image.convert("HSV"), dtype=np.uint8)
        hue_shift = random.randint(-10, 10)
        image_hsv[:, :, 0] = ((image_hsv[:, :, 0].astype(np.int16) + hue_shift) % 256).astype(np.uint8)
        image = Image.fromarray(image_hsv, mode="HSV").convert("RGB")
    if random.random() < 0.4:
        gamma = random.uniform(0.75, 1.35)
        image_array = np.array(image).astype(np.float32) / 255.0
        image_array = np.clip(np.power(image_array, gamma) * 255.0, 0, 255).astype(np.uint8)
        image = Image.fromarray(image_array)
    if random.random() < 0.25:
        image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.2, 1.2)))
    if random.random() < 0.3:
        image_array = np.array(image).astype(np.float32)
        noise = np.random.normal(0.0, random.uniform(2.0, 8.0), image_array.shape)
        image = Image.fromarray(np.clip(image_array + noise, 0, 255).astype(np.uint8))
    if random.random() < 0.4:
        image_array = np.array(image).astype(np.float32)
        height, width = image_array.shape[:2]
        center_x = random.randint(0, width - 1)
        center_y = random.randint(0, height - 1)
        radius_x = random.uniform(width * 0.15, width * 0.45)
        radius_y = random.uniform(height * 0.15, height * 0.45)
        yy, xx = np.ogrid[:height, :width]
        local_mask = ((xx - center_x) / radius_x) ** 2 + ((yy - center_y) / radius_y) ** 2 <= 1.0
        factor = random.uniform(0.55, 0.85) if random.random() < 0.5 else random.uniform(1.15, 1.45)
        image_array[local_mask] = np.clip(image_array[local_mask] * factor, 0, 255)
        image = Image.fromarray(image_array.astype(np.uint8))
    return image


class ForkliftSegmentationDataset(Dataset):
    def __init__(self, records_df: pd.DataFrame, processor: SegformerImageProcessor, training: bool) -> None:
        self.records = records_df.to_dict("records")
        self.processor = processor
        self.training = training

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        record = self.records[index]
        image = Image.open(record["image_path"]).convert("RGB")
        mask_rgb = np.array(Image.open(record["mask_path"]).convert("RGB"), dtype=np.uint8)
        label_mask = Image.fromarray(rgb_mask_to_class_ids(mask_rgb).astype(np.uint8))

        image = image.resize((INPUT_SIZE, INPUT_SIZE), Image.Resampling.BILINEAR)
        label_mask = label_mask.resize((INPUT_SIZE, INPUT_SIZE), Image.Resampling.NEAREST)

        if self.training:
            if random.random() < 0.5:
                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                label_mask = label_mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            image = apply_training_augmentations(image)

        encoded = self.processor(
            images=image,
            segmentation_maps=np.array(label_mask, dtype=np.int64),
            return_tensors="pt",
        )
        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": encoded["labels"].squeeze(0).long(),
        }


records_df = build_annotated_records()
train_records_df, validation_records_df = split_train_validation(records_df)
print("usable annotated image count:", len(records_df))
print("train image count:", len(train_records_df))
print("validation image count:", len(validation_records_df))
display(records_df)


## 4. SegFormer-B0 fine-tune

In [ ]:
def set_backbone_trainable(model: nn.Module, trainable: bool) -> None:
    for name, parameter in model.named_parameters():
        if name.startswith("segformer"):
            parameter.requires_grad = trainable
        else:
            parameter.requires_grad = True


def build_optimizer(model: nn.Module, backbone_trainable: bool) -> torch.optim.Optimizer:
    if not backbone_trainable:
        return torch.optim.AdamW(
            [parameter for parameter in model.parameters() if parameter.requires_grad],
            lr=HEAD_LR,
            weight_decay=WEIGHT_DECAY,
        )

    backbone_parameters = [
        parameter for name, parameter in model.named_parameters()
        if name.startswith("segformer") and parameter.requires_grad
    ]
    head_parameters = [
        parameter for name, parameter in model.named_parameters()
        if not name.startswith("segformer") and parameter.requires_grad
    ]
    return torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": BACKBONE_LR},
            {"params": head_parameters, "lr": HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )


def compute_segmentation_loss(
    model: nn.Module,
    pixel_values: torch.Tensor,
    labels: torch.Tensor,
    class_weights: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    outputs = model(pixel_values=pixel_values)
    logits = F.interpolate(
        outputs.logits,
        size=labels.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )
    loss = F.cross_entropy(logits, labels, weight=class_weights)
    return loss, logits


def mean_iou_from_logits(logits: torch.Tensor, labels: torch.Tensor, num_classes: int) -> dict[str, float]:
    predictions = logits.argmax(dim=1)
    iou_values: list[float] = []
    result: dict[str, float] = {}
    for class_id in range(num_classes):
        prediction_mask = predictions == class_id
        label_mask = labels == class_id
        union = torch.logical_or(prediction_mask, label_mask).sum().item()
        intersection = torch.logical_and(prediction_mask, label_mask).sum().item()
        iou = float("nan") if union == 0 else float(intersection / union)
        result[f"iou_{ID_TO_CLASS[class_id]}"] = iou
        if not math.isnan(iou):
            iou_values.append(iou)
    result["mean_iou"] = float(np.mean(iou_values)) if iou_values else float("nan")
    return result


def run_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    class_weights: torch.Tensor,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_batches = 0
    last_iou: dict[str, float] = {}

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in tqdm(dataloader, leave=False):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            loss, logits = compute_segmentation_loss(model, pixel_values, labels, class_weights)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += float(loss.detach().cpu())
            total_batches += 1
            last_iou = mean_iou_from_logits(logits.detach(), labels, len(CLASS_NAMES))

    metrics = {"loss": total_loss / max(total_batches, 1)}
    metrics.update(last_iou)
    return metrics


if records_df.empty:
    raise ValueError("No annotated SegmentationClass masks are available for fine-tuning.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

processor = SegformerImageProcessor.from_pretrained(
    MODEL_NAME,
    do_resize=False,
    do_reduce_labels=False,
)
model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    id2label={class_id: class_name for class_id, class_name in ID_TO_CLASS.items()},
    label2id=CLASS_TO_ID,
    ignore_mismatched_sizes=True,
).to(device)

class_weights = torch.tensor(
    [CLASS_WEIGHTS[class_name] for class_name in CLASS_NAMES],
    dtype=torch.float32,
    device=device,
)

train_dataset = ForkliftSegmentationDataset(train_records_df, processor, training=True)
validation_dataset = ForkliftSegmentationDataset(validation_records_df, processor, training=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

history_rows: list[dict[str, Any]] = []

set_backbone_trainable(model, trainable=False)
optimizer = build_optimizer(model, backbone_trainable=False)
for epoch in range(1, HEAD_ONLY_EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, class_weights, device, optimizer=optimizer)
    validation_metrics = run_epoch(model, validation_loader, class_weights, device, optimizer=None)
    history_rows.append({"stage": "head_only", "epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in validation_metrics.items()}})
    print(history_rows[-1])

set_backbone_trainable(model, trainable=True)
optimizer = build_optimizer(model, backbone_trainable=True)
for epoch in range(1, FINETUNE_EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, class_weights, device, optimizer=optimizer)
    validation_metrics = run_epoch(model, validation_loader, class_weights, device, optimizer=None)
    history_rows.append({"stage": "fine_tune", "epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in validation_metrics.items()}})
    print(history_rows[-1])

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MODEL_OUTPUT_DIR)
processor.save_pretrained(MODEL_OUTPUT_DIR)
(MODEL_OUTPUT_DIR / "class_config.json").write_text(
    json.dumps({"class_names": CLASS_NAMES, "class_colors": CLASS_COLORS}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
history_df = pd.DataFrame(history_rows)
history_df.to_csv(MODEL_OUTPUT_DIR / "training_history.csv", index=False)
print("saved model to:", MODEL_OUTPUT_DIR)
display(history_df)


## 5. 未アノテーション画像を指定件数推論して保存

このブロックは学習後に繰り返し実行できます。`model` / `processor` がメモリ上にない場合は `MODEL_OUTPUT_DIR` から読み込みます。`movie_preprocess_frames` から、カテゴリ・環境・視点のフィルタ条件を満たす画像をランダムに選んで推論します。`PREDICT_SAVE_TO_PREDICT=True` の場合のみ、マスクと推論対象画像を `ANNOTATION_DIR` をコピーした `PREDICT_DIR` 配下へ保存し、同内容を `PREDICT_ZIP_PATH` に zip として出力します。

In [ ]:
# このセル単体を繰り返し実行できるよう、推論に必要な最小設定をここでも定義します。
from __future__ import annotations

from pathlib import Path
from typing import Any

import random
import shutil
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

try:
    import torch
    import torch.nn.functional as F
    from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "PyTorch / Transformers が見つかりません。先頭の依存パッケージセルを実行してください。"
    ) from error

DATA_DIR = Path(globals().get("DATA_DIR", Path("../data")))
TRAIN_ANNOTATION_BASE_DIR = Path(globals().get("TRAIN_ANNOTATION_BASE_DIR", DATA_DIR / "segmentation" / "train"))


def annotation_zip_path_for_source(annotation_source_path: Path) -> Path:
    if annotation_source_path.suffix == ".zip":
        return annotation_source_path
    return annotation_source_path.parent / f"{annotation_source_path.name}.zip"


def validate_zip_members(zip_path: Path, extract_dir: Path) -> None:
    extract_root = extract_dir.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = (extract_root / member.filename).resolve()
            if not member_path.is_relative_to(extract_root):
                raise ValueError(f"Unsafe zip member path in {zip_path}: {member.filename}")


def extract_annotation_zip(annotation_zip_path: Path, extract_dir: Path) -> Path:
    validate_zip_members(annotation_zip_path, extract_dir)
    source_mtime = str(annotation_zip_path.stat().st_mtime_ns)
    marker_path = extract_dir / ".source_zip_mtime"
    needs_extract = not extract_dir.exists() or not marker_path.exists() or marker_path.read_text(encoding="utf-8") != source_mtime
    if needs_extract:
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(annotation_zip_path) as archive:
            archive.extractall(extract_dir)
        marker_path.write_text(source_mtime, encoding="utf-8")
    return find_annotation_root(extract_dir)


def find_annotation_root(extract_dir: Path) -> Path:
    if (extract_dir / "labelmap.txt").exists() and (extract_dir / "ImageSets" / "Segmentation" / "default.txt").exists():
        return extract_dir
    candidate_roots = [
        path.parent
        for path in extract_dir.rglob("labelmap.txt")
        if (path.parent / "ImageSets" / "Segmentation" / "default.txt").exists()
    ]
    if len(candidate_roots) == 1:
        return candidate_roots[0]
    if not candidate_roots:
        raise FileNotFoundError(f"Could not find CVAT annotation files under: {extract_dir}")
    raise ValueError(f"Multiple annotation roots found under {extract_dir}: {candidate_roots}")


def resolve_annotation_source_path(annotation_dirname: str | Path | None, train_annotation_base_dir: Path) -> Path:
    if annotation_dirname is None:
        zip_candidates = sorted(train_annotation_base_dir.glob("*.zip"))
        if not zip_candidates:
            raise FileNotFoundError(f"No annotation zip files found in: {train_annotation_base_dir}")
        return zip_candidates[0]
    annotation_source_path = Path(annotation_dirname)
    if not annotation_source_path.is_absolute():
        annotation_source_path = train_annotation_base_dir / annotation_source_path
    return annotation_source_path


def resolve_annotation_dir(annotation_source_path: Path) -> tuple[Path, Path | None]:
    source_candidates = [annotation_source_path]
    if annotation_source_path.suffix != ".zip":
        source_candidates.append(annotation_zip_path_for_source(annotation_source_path))
    source_path = next((path for path in source_candidates if path.exists()), None)
    if source_path is None:
        zip_candidates = sorted(annotation_source_path.parent.glob("*.zip"))
        if len(zip_candidates) == 1:
            source_path = zip_candidates[0]
        else:
            raise FileNotFoundError(f"Annotation directory or zip does not exist: {annotation_source_path}")
    if source_path.suffix == ".zip":
        annotation_dir = extract_annotation_zip(source_path, source_path.with_suffix(""))
        return annotation_dir, source_path
    return source_path, annotation_zip_path_for_source(source_path) if annotation_zip_path_for_source(source_path).exists() else None


annotation_dirname = globals().get("ANNOTATION_DIRNANE", None)
annotation_source_default = resolve_annotation_source_path(annotation_dirname, TRAIN_ANNOTATION_BASE_DIR)
annotation_dir_override = globals().get("ANNOTATION_DIR", None)
if annotation_dirname is None and annotation_dir_override is not None and Path(annotation_dir_override).exists():
    ANNOTATION_SOURCE_PATH = Path(annotation_dir_override)
else:
    ANNOTATION_SOURCE_PATH = annotation_source_default
ANNOTATION_DIR, ANNOTATION_ZIP_PATH = resolve_annotation_dir(ANNOTATION_SOURCE_PATH)
IMAGE_DIR = ANNOTATION_DIR / "JPEGImages"
IMAGE_SET_PATH = ANNOTATION_DIR / "ImageSets" / "Segmentation" / "default.txt"
SEGMENTATION_CLASS_DIR = ANNOTATION_DIR / "SegmentationClass"
LABELMAP_PATH = ANNOTATION_DIR / "labelmap.txt"
PREDICT_BASE_DIR = Path(globals().get("PREDICT_BASE_DIR", DATA_DIR / "segmentation" / "predict"))
PREDICT_DIR = PREDICT_BASE_DIR / ANNOTATION_DIR.name
if PREDICT_DIR == PREDICT_BASE_DIR:
    PREDICT_DIR = PREDICT_BASE_DIR / ANNOTATION_DIR.name
PREDICT_ZIP_PATH = PREDICT_BASE_DIR / f"{PREDICT_DIR.name}.zip"
PREDICT_JPEG_IMAGE_DIR = PREDICT_DIR / "JPEGImages"
PREDICT_SEGMENTATION_CLASS_DIR = PREDICT_DIR / "SegmentationClass"
PREDICT_SEGMENTATION_OBJECT_DIR = PREDICT_DIR / "SegmentationObject"
PREDICT_IMAGE_SET_PATH = PREDICT_DIR / "ImageSets" / "Segmentation" / "default.txt"
PREDICT_MANIFEST_PATH = PREDICT_DIR / "prediction_manifest.csv"
MODEL_OUTPUT_DIR = globals().get("MODEL_OUTPUT_DIR", Path("../outputs/segformer_forklift_4class"))
INPUT_SIZE = globals().get("INPUT_SIZE", 512)
OTHER_CLASS_NAME = globals().get("OTHER_CLASS_NAME", "other")

# 推論専用設定はこの単独推論セルで指定します。
PREDICT_IMAGE_DIR = DATA_DIR / "movie_preprocess_frames"
EXCLUDE_ALREADY_PREDICTED = True
PREDICT_RANDOM_SEED = None
PREDICT_IMAGE_FILENAME = None
PREDICT_IMAGE_COUNT = 1
PREDICT_INCLUDE_ANOMARY = True
PREDICT_INCLUDE_NORMAL = True
PREDICT_INCLUDE_INDOOR = True
PREDICT_INCLUDE_OUTDOOR = False
PREDICT_INCLUDE_FRONT = True
PREDICT_INCLUDE_REAR = True
PREDICT_SAVE_TO_PREDICT = False
CLEANUP_WORK_DIRS_AFTER_PREDICT_ZIP = True


def load_labelmap_classes(labelmap_path: Path = LABELMAP_PATH) -> tuple[list[str], dict[str, tuple[int, int, int]]]:
    if not labelmap_path.exists():
        raise FileNotFoundError(f"Labelmap file does not exist: {labelmap_path}")

    class_names: list[str] = []
    class_colors: dict[str, tuple[int, int, int]] = {}
    for line_number, raw_line in enumerate(labelmap_path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        fields = line.split(":")
        if len(fields) < 2:
            raise ValueError(f"Invalid labelmap line {line_number}: {raw_line}")
        class_name = fields[0].strip()
        color_values = tuple(int(value.strip()) for value in fields[1].split(","))
        if not class_name:
            raise ValueError(f"Missing class name in labelmap line {line_number}: {raw_line}")
        if len(color_values) != 3 or any(value < 0 or value > 255 for value in color_values):
            raise ValueError(f"Invalid RGB color in labelmap line {line_number}: {raw_line}")
        if class_name in class_colors:
            raise ValueError(f"Duplicate class name in labelmap: {class_name}")
        class_names.append(class_name)
        class_colors[class_name] = color_values

    if not class_names:
        raise ValueError(f"No classes found in labelmap: {labelmap_path}")
    if OTHER_CLASS_NAME not in class_colors:
        raise ValueError(f"Labelmap must contain '{OTHER_CLASS_NAME}' class: {labelmap_path}")
    return class_names, class_colors


CLASS_NAMES, CLASS_COLORS = load_labelmap_classes()
CLASS_TO_ID = {class_name: class_id for class_id, class_name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {class_id: class_name for class_name, class_id in CLASS_TO_ID.items()}


def read_image_ids_from_path(image_set_path: Path) -> list[str]:
    if not image_set_path.exists():
        return []
    return [line.strip() for line in image_set_path.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_image_ids_to_path(image_set_path: Path, image_ids: list[str]) -> None:
    image_set_path.parent.mkdir(parents=True, exist_ok=True)
    image_set_path.write_text("\n".join(image_ids) + "\n", encoding="utf-8")


def read_default_image_ids(image_set_path: Path = IMAGE_SET_PATH) -> list[str]:
    if not image_set_path.exists():
        raise FileNotFoundError(f"Image set file does not exist: {image_set_path}")
    return read_image_ids_from_path(image_set_path)


def extract_predict_zip_if_needed() -> None:
    if PREDICT_DIR.exists() or not PREDICT_ZIP_PATH.exists():
        return
    validate_zip_members(PREDICT_ZIP_PATH, PREDICT_DIR)
    PREDICT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PREDICT_ZIP_PATH) as archive:
        archive.extractall(PREDICT_DIR)
    if PREDICT_SEGMENTATION_OBJECT_DIR.exists():
        shutil.rmtree(PREDICT_SEGMENTATION_OBJECT_DIR)


def write_predict_annotation_zip() -> Path:
    if not PREDICT_DIR.exists():
        raise FileNotFoundError(f"Predict annotation directory does not exist: {PREDICT_DIR}")
    PREDICT_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_zip_path = PREDICT_ZIP_PATH.with_name(f"{PREDICT_ZIP_PATH.name}.tmp")
    if temporary_zip_path.exists():
        temporary_zip_path.unlink()
    with zipfile.ZipFile(temporary_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for source_path in sorted(PREDICT_DIR.rglob("*")):
            if source_path.is_dir():
                continue
            relative_path = source_path.relative_to(PREDICT_DIR)
            if relative_path.name in {".source_zip_mtime", "prediction_manifest.csv"}:
                continue
            if relative_path.parts and relative_path.parts[0] == "SegmentationObject":
                continue
            archive.write(source_path, relative_path.as_posix())
    temporary_zip_path.replace(PREDICT_ZIP_PATH)
    return PREDICT_ZIP_PATH


def cleanup_annotation_work_dirs() -> list[Path]:
    removed_paths: list[Path] = []
    if PREDICT_ZIP_PATH.exists() and PREDICT_DIR.exists():
        shutil.rmtree(PREDICT_DIR)
        removed_paths.append(PREDICT_DIR)
    if ANNOTATION_ZIP_PATH is not None:
        annotation_work_dir = ANNOTATION_ZIP_PATH.with_suffix("")
        if annotation_work_dir.exists() and (annotation_work_dir / ".source_zip_mtime").exists():
            shutil.rmtree(annotation_work_dir)
            removed_paths.append(annotation_work_dir)
    return removed_paths


def prediction_image_path_for_id(image_id: str) -> Path:
    return PREDICT_IMAGE_DIR / f"{image_id}.png"


def resolve_prediction_image_path(image_filename: str | Path) -> Path:
    image_path = PREDICT_IMAGE_DIR / Path(image_filename).name
    if image_path.suffix == "":
        image_path = image_path.with_suffix(".png")
    if image_path.exists():
        return image_path
    source_image_path = prediction_image_path_for_id(canonical_source_image_id(image_path.stem))
    if source_image_path.exists():
        return source_image_path
    raise FileNotFoundError(f"Prediction image does not exist: {image_path}")


def canonical_source_image_id(image_id: str) -> str:
    parts = image_id.split("_", 4)
    if (
        len(parts) == 5
        and parts[0] in {"normal", "anomary"}
        and parts[1] in {"indoor", "outdoor"}
        and parts[2] in {"front", "rear"}
        and parts[3].isdigit()
        and len(parts[3]) == 6
    ):
        return f"{parts[0]}_{parts[1]}_{parts[2]}_{parts[4]}"
    return image_id


def copy_annotation_dir_to_predict(preserve_existing_segmentation_masks: bool = True) -> None:
    if not ANNOTATION_DIR.exists():
        raise FileNotFoundError(f"Annotation directory does not exist: {ANNOTATION_DIR}")
    extract_predict_zip_if_needed()
    for source_path in sorted(ANNOTATION_DIR.rglob("*")):
        relative_path = source_path.relative_to(ANNOTATION_DIR)
        if relative_path.name == ".source_zip_mtime":
            continue
        if relative_path.parts and relative_path.parts[0] == "SegmentationObject":
            continue
        target_path = PREDICT_DIR / relative_path
        if source_path.is_dir():
            target_path.mkdir(parents=True, exist_ok=True)
            continue
        if (
            preserve_existing_segmentation_masks
            and target_path.exists()
            and relative_path.parts
            and relative_path.parts[0] in {"SegmentationClass", "SegmentationObject"}
        ):
            continue
        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, target_path)
    if PREDICT_SEGMENTATION_OBJECT_DIR.exists():
        shutil.rmtree(PREDICT_SEGMENTATION_OBJECT_DIR)


def prepare_predict_annotation_dir(preserve_existing_segmentation_masks: bool = True) -> None:
    extract_predict_zip_if_needed()
    existing_predict_ids = read_image_ids_from_path(PREDICT_IMAGE_SET_PATH)
    copy_annotation_dir_to_predict(preserve_existing_segmentation_masks=preserve_existing_segmentation_masks)
    merged_image_ids = read_image_ids_from_path(PREDICT_IMAGE_SET_PATH)
    for image_id in existing_predict_ids:
        if image_id not in merged_image_ids:
            merged_image_ids.append(image_id)
    if merged_image_ids:
        write_image_ids_to_path(PREDICT_IMAGE_SET_PATH, merged_image_ids)
    write_predict_annotation_zip()


def class_ids_to_rgb_mask(class_id_mask: np.ndarray) -> np.ndarray:
    rgb_mask = np.zeros((*class_id_mask.shape, 3), dtype=np.uint8)
    for class_id, class_name in ID_TO_CLASS.items():
        rgb_mask[class_id_mask == class_id] = np.array(CLASS_COLORS[class_name], dtype=np.uint8)
    return rgb_mask


def update_predict_image_set(image_id_or_filename: str) -> None:
    image_id = Path(image_id_or_filename).stem
    image_ids = read_image_ids_from_path(PREDICT_IMAGE_SET_PATH)
    if image_id not in image_ids:
        image_ids.append(image_id)
        write_image_ids_to_path(PREDICT_IMAGE_SET_PATH, image_ids)


def copy_prediction_image_to_predict_jpeg_images(source_image_path: Path, image_id_or_filename: str) -> Path:
    target_image_path = PREDICT_JPEG_IMAGE_DIR / f"{Path(image_id_or_filename).stem}{source_image_path.suffix or '.png'}"
    PREDICT_JPEG_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    if source_image_path.resolve() != target_image_path.resolve():
        shutil.copy2(source_image_path, target_image_path)
    return target_image_path


def load_trained_model_for_inference() -> tuple[SegformerForSemanticSegmentation, SegformerImageProcessor, torch.device]:
    inference_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if "model" in globals() and "processor" in globals():
        inference_model = model.to(inference_device)
        inference_processor = processor
    else:
        if not MODEL_OUTPUT_DIR.exists():
            raise FileNotFoundError(f"Trained model directory does not exist: {MODEL_OUTPUT_DIR}")
        inference_model = SegformerForSemanticSegmentation.from_pretrained(MODEL_OUTPUT_DIR).to(inference_device)
        inference_processor = SegformerImageProcessor.from_pretrained(MODEL_OUTPUT_DIR, do_resize=False)
    inference_model.eval()
    return inference_model, inference_processor, inference_device


def parse_prediction_image_metadata(image_path: Path) -> dict[str, str] | None:
    parts = image_path.stem.split("_", 3)
    if len(parts) < 4:
        return None
    category, environment, view_name = parts[:3]
    if category not in {"anomary", "normal"}:
        return None
    if environment not in {"indoor", "outdoor"}:
        return None
    if view_name not in {"front", "rear"}:
        return None
    return {"category": category, "environment": environment, "view_name": view_name}


def prediction_metadata_is_enabled(metadata: dict[str, str]) -> bool:
    category_enabled = (
        (metadata["category"] == "anomary" and PREDICT_INCLUDE_ANOMARY)
        or (metadata["category"] == "normal" and PREDICT_INCLUDE_NORMAL)
    )
    environment_enabled = (
        (metadata["environment"] == "indoor" and PREDICT_INCLUDE_INDOOR)
        or (metadata["environment"] == "outdoor" and PREDICT_INCLUDE_OUTDOOR)
    )
    view_enabled = (
        (metadata["view_name"] == "front" and PREDICT_INCLUDE_FRONT)
        or (metadata["view_name"] == "rear" and PREDICT_INCLUDE_REAR)
    )
    return category_enabled and environment_enabled and view_enabled


def collect_prediction_candidates_from_frames() -> list[dict[str, Any]]:
    if not PREDICT_IMAGE_DIR.exists():
        raise FileNotFoundError(f"Prediction image directory does not exist: {PREDICT_IMAGE_DIR}")
    predicted_ids = {path.stem for path in PREDICT_SEGMENTATION_CLASS_DIR.glob("*.png")} if EXCLUDE_ALREADY_PREDICTED else set()
    candidates = []
    for image_path in sorted(PREDICT_IMAGE_DIR.glob("*.png")):
        metadata = parse_prediction_image_metadata(image_path)
        if metadata is None or not prediction_metadata_is_enabled(metadata):
            continue
        image_id = image_path.stem
        if image_id in predicted_ids:
            continue
        candidates.append({
            "image_id": image_id,
            "source_image_id": image_id,
            "source_image_path": image_path,
            **metadata,
        })
    return candidates


def predict_class_id_mask(
    image: Image.Image,
    inference_model: SegformerForSemanticSegmentation,
    inference_processor: SegformerImageProcessor,
    inference_device: torch.device,
) -> np.ndarray:
    original_width, original_height = image.size
    resized_image = image.resize((INPUT_SIZE, INPUT_SIZE), Image.Resampling.BILINEAR)
    inputs = inference_processor(images=resized_image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(inference_device)
    with torch.no_grad():
        outputs = inference_model(pixel_values=pixel_values)
        logits = F.interpolate(
            outputs.logits,
            size=(original_height, original_width),
            mode="bilinear",
            align_corners=False,
        )
    return logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)


def overlay_prediction(image_rgb: np.ndarray, mask_rgb: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    return np.clip(image_rgb.astype(np.float32) * (1.0 - alpha) + mask_rgb.astype(np.float32) * alpha, 0, 255).astype(np.uint8)


def append_prediction_manifest(rows: list[dict[str, Any]]) -> pd.DataFrame:
    new_rows_df = pd.DataFrame(rows)
    if PREDICT_MANIFEST_PATH.exists():
        manifest_df = pd.read_csv(PREDICT_MANIFEST_PATH)
        manifest_df = manifest_df.drop(columns=["segmentation_object_path"], errors="ignore")
        manifest_df = pd.concat([manifest_df, new_rows_df], ignore_index=True)
    else:
        manifest_df = new_rows_df
    PREDICT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    manifest_df.to_csv(PREDICT_MANIFEST_PATH, index=False)
    return manifest_df


if PREDICT_IMAGE_COUNT <= 0:
    raise ValueError("PREDICT_IMAGE_COUNT must be greater than 0.")

if PREDICT_SAVE_TO_PREDICT:
    prepare_predict_annotation_dir()

inference_rng = random.Random(PREDICT_RANDOM_SEED)
if PREDICT_IMAGE_FILENAME is None:
    candidates = collect_prediction_candidates_from_frames()
    print("movie_preprocess_frames candidate count:", len(candidates))
    if not candidates:
        raise ValueError("No images match the prediction filters.")
    selected_count = min(PREDICT_IMAGE_COUNT, len(candidates))
    if selected_count < PREDICT_IMAGE_COUNT:
        print(f"Requested {PREDICT_IMAGE_COUNT} images, but only {selected_count} candidates are available.")
    selected_candidates = inference_rng.sample(candidates, selected_count)
else:
    selected_image_path = resolve_prediction_image_path(PREDICT_IMAGE_FILENAME)
    metadata = parse_prediction_image_metadata(selected_image_path) or {
        "category": "",
        "environment": "",
        "view_name": "",
    }
    selected_candidates = [{
        "image_id": selected_image_path.stem,
        "source_image_id": selected_image_path.stem,
        "source_image_path": selected_image_path,
        **metadata,
    }]

inference_model, inference_processor, inference_device = load_trained_model_for_inference()
prediction_rows: list[dict[str, Any]] = []
preview_payload: tuple[np.ndarray, np.ndarray, np.ndarray] | None = None

for selected_candidate in selected_candidates:
    selected_image_id = selected_candidate["image_id"]
    selected_source_image_id = selected_candidate["source_image_id"]
    selected_image_path = selected_candidate["source_image_path"]
    selected_image = Image.open(selected_image_path).convert("RGB")
    predicted_class_id_mask = predict_class_id_mask(
        selected_image,
        inference_model,
        inference_processor,
        inference_device,
    )
    predicted_mask_rgb = class_ids_to_rgb_mask(predicted_class_id_mask)
    predicted_class_path = PREDICT_SEGMENTATION_CLASS_DIR / f"{selected_image_id}.png"
    annotation_image_path: Path | None = None

    if PREDICT_SAVE_TO_PREDICT:
        PREDICT_SEGMENTATION_CLASS_DIR.mkdir(parents=True, exist_ok=True)
        Image.fromarray(predicted_mask_rgb).save(predicted_class_path)
        annotation_image_path = copy_prediction_image_to_predict_jpeg_images(selected_image_path, selected_image_id)
        update_predict_image_set(selected_image_id)

    class_counts = {
        class_name: int((predicted_class_id_mask == class_id).sum())
        for class_id, class_name in ID_TO_CLASS.items()
    }
    row = {
        "image_id": selected_image_id,
        "source_image_id": selected_source_image_id,
        "image_path": str(selected_image_path),
        "category": selected_candidate.get("category", ""),
        "environment": selected_candidate.get("environment", ""),
        "view_name": selected_candidate.get("view_name", ""),
        "save_to_predict": PREDICT_SAVE_TO_PREDICT,
        "predict_dir": str(PREDICT_DIR) if PREDICT_SAVE_TO_PREDICT else None,
        "segmentation_class_path": str(predicted_class_path) if PREDICT_SAVE_TO_PREDICT else None,
        "jpeg_image_path": str(annotation_image_path) if annotation_image_path is not None else None,
        "model_output_dir": str(MODEL_OUTPUT_DIR),
        **{f"predicted_{class_name}_pixel_count": count for class_name, count in class_counts.items()},
    }
    prediction_rows.append(row)
    if preview_payload is None:
        preview_image_rgb = np.array(selected_image, dtype=np.uint8)
        preview_overlay_rgb = overlay_prediction(preview_image_rgb, predicted_mask_rgb)
        preview_payload = (preview_image_rgb, predicted_mask_rgb, preview_overlay_rgb)

if PREDICT_SAVE_TO_PREDICT:
    append_prediction_manifest(prediction_rows)
    write_predict_annotation_zip()

prediction_results_df = pd.DataFrame(prediction_rows)
print("selected image count:", len(prediction_results_df))
print("save to predict:", PREDICT_SAVE_TO_PREDICT)
if PREDICT_SAVE_TO_PREDICT:
    print("prediction annotation dir:", PREDICT_DIR)
    print("prediction annotation zip:", PREDICT_ZIP_PATH)
display(prediction_results_df)

if preview_payload is not None:
    print("preview image id:", prediction_results_df.iloc[0]["image_id"])
    preview_image_rgb, predicted_mask_rgb, preview_overlay_rgb = preview_payload
    display(Image.fromarray(preview_image_rgb))
    display(Image.fromarray(predicted_mask_rgb))
    display(Image.fromarray(preview_overlay_rgb))

if PREDICT_SAVE_TO_PREDICT and CLEANUP_WORK_DIRS_AFTER_PREDICT_ZIP:
    removed_work_dirs = cleanup_annotation_work_dirs()
    if removed_work_dirs:
        print("removed annotation work dirs:", [str(path) for path in removed_work_dirs])
